# cracksteam456 Voice Studio — Thử nghiệm 10 giọng tiếng Việt
Chạy ô cài đặt, Restart session, sau đó chạy ô model và ô thử nghiệm.
Dùng preview gốc để đánh giá kỹ thuật; quyền clone ngoài ElevenLabs chưa xác minh. Kết quả không tự được thêm vào thư viện phát hành. Mẫu được chọn theo mốc câu từ Whisper, cần nghe kiểm tra lại.


In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'omnivoice==0.2.1', 'gradio', 'numpy==2.1.3', 'scipy==1.15.3', 'requests==2.32.4'], check=True)

# Verify native imports in a fresh process; the notebook kernel may still hold old modules.
subprocess.run([sys.executable, '-c', 'import numpy, scipy.linalg; print("NumPy/SciPy: OK")'], check=True)

from pathlib import Path
from urllib.request import urlretrieve
sample_url = 'https://raw.githubusercontent.com/doanquangkien/voice-notebooks/main/samples/nam-cong-nghe.mp3'
urlretrieve(sample_url, 'voice_sample.mp3')
assert Path('voice_sample.mp3').stat().st_size > 0
print('Đã cài thư viện và tải giọng mẫu. Chọn Runtime → Restart session, rồi chạy ô khởi động bên dưới.')

In [ ]:
from importlib.metadata import version
import sys
for package in ('numpy', 'scipy'):
    loaded = sys.modules.get(package)
    if loaded is not None and loaded.__version__ != version(package):
        raise RuntimeError('Thư viện cũ còn trong bộ nhớ. Chọn Runtime → Restart session, rồi chạy lại ô này.')
import scipy.linalg
import logging
import re
import numpy as np
import torch
import gradio as gr

# Compatibility shims for newer Colab PyTorch and Transformers releases.
if not hasattr(torch, '_utils'):
    torch._utils = torch._C._utils

import transformers as _tf
class _SafeAutoFeatureExtractor:
    @staticmethod
    def from_pretrained(model_name, **kwargs):
        try:
            from transformers import AutoConfig
            cfg = AutoConfig.from_pretrained(model_name, trust_remote_code=True, **kwargs)
            sample_rate = getattr(cfg, 'sampling_rate', 24000)
        except Exception:
            sample_rate = 24000
        from types import SimpleNamespace
        return SimpleNamespace(sampling_rate=sample_rate)
_tf.AutoFeatureExtractor = _SafeAutoFeatureExtractor

from omnivoice import OmniVoice, OmniVoiceGenerationConfig
from omnivoice.utils.common import get_best_device

if not torch.cuda.is_available():
    raise RuntimeError('Chưa có GPU. Chọn Runtime → Change runtime type → T4 GPU rồi chạy lại.')
print('GPU:', torch.cuda.get_device_name(0))

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')
device = get_best_device()
print('Đang tải OmniVoice lên', device, '...')
if 'model' not in globals():
    model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map=device, dtype=torch.float16, load_asr=True)
sampling_rate = model.sampling_rate


In [ ]:
CANDIDATES = [{'name': 'Tuan Anh - Deep & Powerful Vietnamese', 'voice_id': '47rM9DW5VmjOw23BVHWi', 'public_owner_id': 'a754c5fd4e5a50305d5e8e8aa356b03e6aeb7c63f4e906334d2e07023680a973', 'primary_language': 'vi', 'rank': 1, 'preview_url': 'https://storage.googleapis.com/eleven-public-prod/database/workspace/8b673ecd50964d0c9a44fd4be3ee4e75/voices/47rM9DW5VmjOw23BVHWi/k5S2vEYnPfG1H9GBorFI.mp3', 'usage_character_count_1y': 18049361, 'clone_permission': 'unknown', 'audio_quality': 'not_auditioned', 'folder': 'voice-01', 'quality_status': 'not_auditioned', 'reference_selection': 'pending_asr_sentence_boundaries', 'original_sha256': 'ca771233a725a4b64ef4c22f8de73aa1baeac7c590fa69fa8e59a8466f2b7814', 'original_duration': 7.758313, 'status': 'preview_downloaded'}, {'name': 'Mai Anh - Vietnamese News Anchor', 'voice_id': 'Hq7udsaTAvCBrAg8DSP2', 'public_owner_id': '910f350f80b4d4588bfed5d10db52fc5fd690d510c75ee425f7056d8670a6ef7', 'primary_language': 'vi', 'rank': 2, 'preview_url': 'https://storage.googleapis.com/eleven-public-prod/database/workspace/488d3fc4c4a0413693686c63387fc0a5/voices/Hq7udsaTAvCBrAg8DSP2/xgeIyK0bPA26T74thTHr.mp3', 'usage_character_count_1y': 964146, 'clone_permission': 'unknown', 'audio_quality': 'not_auditioned', 'folder': 'voice-02', 'quality_status': 'not_auditioned', 'reference_selection': 'pending_asr_sentence_boundaries', 'original_sha256': '039c529da3c9631e5da4531861c1f9c9905083f8e33780f7a4c3d7313d58538d', 'original_duration': 9.168938, 'status': 'preview_downloaded'}, {'name': 'Thieuthattha - Casual', 'voice_id': 'VnlraNGkeLNUIQm7PPQ7', 'public_owner_id': '38ad5edef16cf8491c789caf545660381486e0acdf6bb7899e9f2bfd8532d24f', 'primary_language': 'vi', 'rank': 3, 'preview_url': 'https://storage.googleapis.com/eleven-public-prod/database/workspace/19fd6cab37114025bbb1fdcd3877d23c/voices/VnlraNGkeLNUIQm7PPQ7/71ecc016-391e-4f0f-80a5-d1cb122a5186.mp3', 'usage_character_count_1y': 74567, 'clone_permission': 'unknown', 'audio_quality': 'not_auditioned', 'folder': 'voice-03', 'quality_status': 'not_auditioned', 'reference_selection': 'pending_asr_sentence_boundaries', 'original_sha256': '14656bf9751ada5d54a0abac79a9ee8dde70ab7b2513de1b2f578486b4c8bf57', 'original_duration': 4.8, 'status': 'preview_downloaded'}, {'name': 'Quang Toan - Deep Vietnamese Storyteller', 'voice_id': 'ZsjEJaLQy3sgvwxicmDx', 'public_owner_id': '5c3a5371161ed2a7a4b3ed617dad7247ff46a4c9a2d4f9e7af18fb635e971dc6', 'primary_language': 'vi', 'rank': 4, 'preview_url': 'https://storage.googleapis.com/eleven-public-prod/database/workspace/d34501e492854e51876b1e466f1e9254/voices/ZsjEJaLQy3sgvwxicmDx/ArqPV7irBkPMVFekCK5q.mp3', 'usage_character_count_1y': 19959890, 'clone_permission': 'unknown', 'audio_quality': 'not_auditioned', 'folder': 'voice-04', 'quality_status': 'not_auditioned', 'reference_selection': 'pending_asr_sentence_boundaries', 'original_sha256': '2267c80fee37b744c4a4be06fdebacf2bee15f45e1776b9a8c7499e4d5db30c0', 'original_duration': 7.000813, 'status': 'preview_downloaded'}, {'name': 'Ba Luc - Warm, Clear, Storytelling', 'voice_id': 'KJmTHSk3TjHp87WObL98', 'public_owner_id': 'd313f0d270a4cb5ff44c22e4ee308206a3801e424de4e377bb203c7e6d6d92ba', 'primary_language': 'vi', 'rank': 5, 'preview_url': 'https://api.us.elevenlabs.io/v1/voices/KJmTHSk3TjHp87WObL98/previews/audio?payload=eyJ2b2ljZV9zb3VyY2UiOiJjdXN0b20iLCJ3b3Jrc3BhY2VfaWQiOiJlNjQ2Y2Q0NjVhYmU0M2ZmODY4ZTAxOTFiZmE4NDcyYyIsImZpbGVuYW1lIjoiTU0zZWRYUTZDcFNKYTBweUpsQ3IubXAzIiwidGltZXN0YW1wIjoxNzk0OTA5NjAwMDAwMDAwfQ%3D%3D', 'usage_character_count_1y': 10713635, 'clone_permission': 'unknown', 'audio_quality': 'not_auditioned', 'folder': 'voice-05', 'quality_status': 'not_auditioned', 'reference_selection': 'pending_asr_sentence_boundaries', 'original_sha256': 'beb9f93b4d9ae457a48520af44b86ca816dedbff1f2102466950c0ca080a5147', 'original_duration': 9.7175, 'status': 'preview_downloaded'}, {'name': 'Thảo - Soft, Calm and Gentle', 'voice_id': '558B1EcdabtcSdleer40', 'public_owner_id': 'f456d3ede4a8033c711858c1dd921b89dfb93bffab789be9a29759a2fddf3b6a', 'primary_language': 'vi', 'rank': 6, 'preview_url': 'https://storage.googleapis.com/eleven-public-prod/database/user/DxkgIq1vnwhTYa8uL32YhmxjMou2/voices/558B1EcdabtcSdleer40/3gFzJJf8SfUG0fPjLHVQ.mp3', 'usage_character_count_1y': 23837974, 'clone_permission': 'unknown', 'audio_quality': 'not_auditioned', 'folder': 'voice-06', 'quality_status': 'not_auditioned', 'reference_selection': 'pending_asr_sentence_boundaries', 'original_sha256': '9d041e6f19efe96afc53aadb2cd98954f04ff9118f0642252139c88eba63a78b', 'original_duration': 7.340375, 'status': 'preview_downloaded'}, {'name': 'Giang - Podcast', 'voice_id': 'CQAD6iKxS73fEAGjSwt5', 'public_owner_id': '558052bb2b6b8782ebdcef34783117b6ac96d0a0ffe084bddb8327aabe89f50b', 'primary_language': 'vi', 'rank': 7, 'preview_url': 'https://storage.googleapis.com/eleven-public-prod/database/workspace/56decef8b5a648b7ab29b17eefba2fcd/voices/CQAD6iKxS73fEAGjSwt5/UmsJG3aeWYxO76OE1zmv.mp3', 'usage_character_count_1y': 10976516, 'clone_permission': 'unknown', 'audio_quality': 'not_auditioned', 'folder': 'voice-07', 'quality_status': 'not_auditioned', 'reference_selection': 'pending_asr_sentence_boundaries', 'original_sha256': '1081140fc17140e7aec3ef1a8f35c001b124fced00638316e3e417c4ee91e482', 'original_duration': 6.164875, 'status': 'preview_downloaded'}, {'name': 'Trieu Duong - Deep, Calm and Resonant', 'voice_id': 'UsgbMVmY3U59ijwK5mdh', 'public_owner_id': '93ae4e172cf1ef9b1a2e8ec7da1bc2d041d5913d5fa93852e42dc3e6c4ae7b1d', 'primary_language': 'vi', 'rank': 8, 'preview_url': 'https://api.us.elevenlabs.io/v1/voices/UsgbMVmY3U59ijwK5mdh/previews/audio?payload=eyJ2b2ljZV9zb3VyY2UiOiJjdXN0b20iLCJ3b3Jrc3BhY2VfaWQiOiJhODZjMjc1ZmQyMTk0ZjcwOTFkMGMyNjdmYWVmYmRhYyIsImZpbGVuYW1lIjoiZjkwMGEwNDAtMGFiZS00N2RiLWFlYjQtYzIwYjYzMDU2NDMzLm1wMyIsInRpbWVzdGFtcCI6MTc5NDkwOTYwMDAwMDAwMH0%3D', 'usage_character_count_1y': 167322071, 'clone_permission': 'unknown', 'audio_quality': 'not_auditioned', 'folder': 'voice-08', 'quality_status': 'not_auditioned', 'reference_selection': 'pending_asr_sentence_boundaries', 'original_sha256': '7b9992cecf7ca79dce62cc989642bf506f26010f5f654e310e0bf04dc0cba634', 'original_duration': 3.291425, 'status': 'preview_downloaded'}, {'name': 'Hạ Vi - Calm & Trustworthy', 'voice_id': 'iSV1G4KvpGE3D8qzAzwG', 'public_owner_id': 'ba70ff5aac7f11e730cda7c068d118d4449b292e3db81e433ec3d5640c455871', 'primary_language': 'vi', 'rank': 9, 'preview_url': 'https://storage.googleapis.com/eleven-public-prod/database/workspace/3020b36341374893b5e081711e4812e8/voices/iSV1G4KvpGE3D8qzAzwG/uL97ZUIacQXPSGfrR4j1.mp3', 'usage_character_count_1y': 3880576, 'clone_permission': 'unknown', 'audio_quality': 'not_auditioned', 'folder': 'voice-09', 'quality_status': 'not_auditioned', 'reference_selection': 'pending_asr_sentence_boundaries', 'original_sha256': '29684a3120d5889694b02be551e27559efac0736fbac40e926559143317c57c7', 'original_duration': 7.653875, 'status': 'preview_downloaded'}, {'name': 'Gia Huy - Southern TV Host', 'voice_id': '2gohBomXaprQ214IIpWp', 'public_owner_id': 'e9297c45a8602324cb5e70cf43f14f94f245baede48bba48c1cc69de19654cf5', 'primary_language': 'vi', 'rank': 10, 'preview_url': 'https://storage.googleapis.com/eleven-public-prod/database/workspace/4c2c3bf588984b89a5ce0b460f707550/voices/2gohBomXaprQ214IIpWp/TxtKDNPQFtWpgEs4MWjc.mp3', 'usage_character_count_1y': 74888, 'clone_permission': 'unknown', 'audio_quality': 'not_auditioned', 'folder': 'voice-10', 'quality_status': 'not_auditioned', 'reference_selection': 'pending_asr_sentence_boundaries', 'original_sha256': '3ad8a5f1421ed37c23249f7d1c7b46d2724e04434e31ea80971e3e445d1022d6', 'original_duration': 8.986063, 'status': 'preview_downloaded'}]

"""Run inside the pilot notebook after loading `model`; CANDIDATES is embedded."""
from pathlib import Path
import hashlib
import importlib.metadata
import json
import subprocess
import time
import zipfile
from urllib.request import Request, HTTPRedirectHandler, build_opener
from urllib.parse import urlsplit
import numpy as np
import soundfile as sf
import torch

TEXTS = [
    'Hôm nay, chúng ta tìm hiểu cách âm thanh truyền qua không khí. Mỗi dao động nhỏ đều có thể mang theo một thông điệp.',
    'Nếu Mặt Trăng biến mất, điều gì sẽ xảy ra với thủy triều? Hãy cùng xem những thay đổi đầu tiên trên Trái Đất.',
]
SEEDS = [42, 123]
DEST = Path('/content/clone-pilot')
DEST.mkdir(exist_ok=True)


def select_windows(chunks):
    """Prefer whole ASR sentence spans, never invent cut points within words."""
    windows = []
    for start in range(len(chunks)):
        first = chunks[start].get('timestamp', [None, None])[0]
        if first is None:
            continue
        for end in range(start, len(chunks)):
            last = chunks[end].get('timestamp', [None, None])[1]
            if last is None:
                continue
            duration = last - first
            if duration > 10:
                break
            if 3 <= duration <= 10:
                text = ' '.join(c['text'].strip() for c in chunks[start:end + 1]).strip()
                if text:
                    windows.append({'start': first, 'end': last, 'text': text})
    windows.sort(key=lambda x: abs(x['end'] - x['start'] - 7))
    selected = []
    for window in windows:
        if all(window['end'] <= old['start'] or window['start'] >= old['end'] for old in selected):
            selected.append(window)
        if len(selected) == 2:
            break
    return selected


config = OmniVoiceGenerationConfig(num_step=32, guidance_scale=1.8,
    denoise=True, preprocess_prompt=True, postprocess_output=True,
    position_temperature=5.0, class_temperature=0.2, pad_duration=0.1, fade_duration=0.1)
report = {'package_version': importlib.metadata.version('omnivoice'),
          'model': 'k2-fsa/OmniVoice', 'sample_rate': model.sampling_rate,
          'model_revision': getattr(model.config, '_commit_hash', None),
          'generation_config': vars(config),
          'gpu': torch.cuda.get_device_name(0), 'texts': TEXTS, 'seeds': SEEDS,
          'clone_permission': 'unknown', 'voices': []}

for number, candidate in enumerate(CANDIDATES, 1):
    folder = DEST / candidate['folder']
    folder.mkdir(exist_ok=True)
    result = {**candidate, 'outputs': [], 'reference_review': 'needs_listening'}
    try:
        if candidate.get('status') != 'preview_downloaded':
            raise ValueError('Preview did not pass local preparation.')
        original = folder / 'preview.mp3'
        if not original.exists():
            parsed = urlsplit(candidate['preview_url'])
            if parsed.scheme != 'https' or parsed.hostname not in {'storage.googleapis.com', 'api.us.elevenlabs.io'}:
                raise ValueError('Unapproved preview URL.')
            class NoRedirect(HTTPRedirectHandler):
                def redirect_request(self, *args, **kwargs):
                    return None
            with build_opener(NoRedirect()).open(Request(candidate['preview_url']), timeout=30) as response:
                data = response.read(25_000_001)
            if not data or len(data) > 25_000_000:
                raise ValueError('Preview file size is invalid.')
            original.write_bytes(data)
        digest = hashlib.sha256(original.read_bytes()).hexdigest()
        if digest != candidate['original_sha256']:
            raise ValueError('Preview changed since preparation; choose a fresh reference.')
        transcription_path = folder / 'transcription.json'
        if transcription_path.exists():
            transcription = json.loads(transcription_path.read_text())
        else:
            transcription = model._asr_pipe(str(original), return_timestamps=True,
                generate_kwargs={'language': 'vi', 'task': 'transcribe'})
            transcription_path.write_text(json.dumps(transcription, ensure_ascii=False, indent=2), encoding='utf-8')
        windows = select_windows(transcription.get('chunks', []))
        if not windows:
            raise ValueError('No complete ASR segment of 3–10s; needs manual reference selection.')
        result['references'] = windows
        for ref_index, window in enumerate(windows, 1):
            ref = folder / f'reference-{ref_index}.wav'
            subprocess.run(['ffmpeg', '-y', '-v', 'error', '-ss', str(window['start']),
                '-i', str(original), '-t', str(window['end'] - window['start']),
                '-ac', '1', '-ar', str(model.sampling_rate), '-c:a', 'pcm_s16le', str(ref)], check=True)
            (folder / f'reference-{ref_index}.txt').write_text(window['text'], encoding='utf-8')
            prompt = model.create_voice_clone_prompt(ref_audio=str(ref), ref_text=window['text'])
            prompt.save(str(folder / f'prompt-{ref_index}.pt'))
            for text_index, text in enumerate(TEXTS, 1):
                for seed in SEEDS:
                    out = folder / f'ref-{ref_index}-text-{text_index}-seed-{seed}.wav'
                    started = time.monotonic()
                    if not out.exists():
                        torch.manual_seed(seed)
                        audio = np.asarray(model.generate(text=text, language='vi',
                            voice_clone_prompt=prompt, speed=1.0, generation_config=config)[0])
                        if not audio.size or not np.isfinite(audio).all():
                            raise ValueError('Invalid generated waveform.')
                        sf.write(out, np.clip(audio, -1, 1), model.sampling_rate, subtype='PCM_16')
                    info = sf.info(out)
                    result['outputs'].append({'file': out.name, 'duration': info.duration,
                        'reference': ref_index, 'text_index': text_index, 'seed': seed,
                        'processing_seconds': time.monotonic() - started})
        result['status'] = 'generated_needs_listening'
    except Exception as error:
        result.update(status='failed', error=f'{type(error).__name__}: {error}')
    report['voices'].append(result)
    (folder / 'result.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    (DEST / 'report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'{number}/{len(CANDIDATES)}: {candidate["name"]} — {result["status"]}', flush=True)

archive = Path('/content/clone-pilot-results.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for path in DEST.rglob('*'):
        if path.is_file() and path.suffix != '.pt':
            bundle.write(path, str(path.relative_to(DEST)))
print('PILOT_COMPLETE', sum(len(v['outputs']) for v in report['voices']), 'WAV outputs', flush=True)
from IPython.display import FileLink, display
display(FileLink(str(archive)))
import gradio as gr
with gr.Blocks(title='Kết quả thử clone') as pilot_results:
    gr.Markdown('# Kết quả thử clone\nTải báo cáo và âm thanh để nghe đối chiếu. Chưa chấm chất lượng giọng.')
    gr.File(value=[str(archive), str(DEST / 'report.json')], label='Tải kết quả', interactive=False)
pilot_results.launch(share=True, debug=False)


## Kiểm tra lời đọc bằng Whisper
Chạy sau bước tổng hợp. WER tự động không phải điểm giống giọng; vẫn cần nghe đối chiếu.

In [ ]:
"""Whisper transcript check; run in the same Colab runtime as the pilot."""
from pathlib import Path
import json
import re
import unicodedata
import zipfile


def normalized_words(text):
    text = unicodedata.normalize('NFC', text).casefold()
    return re.findall(r'\w+', text, flags=re.UNICODE)


def edit_distance(expected, actual):
    previous = list(range(len(actual) + 1))
    for i, left in enumerate(expected, 1):
        current = [i]
        for j, right in enumerate(actual, 1):
            current.append(min(current[-1] + 1, previous[j] + 1,
                               previous[j - 1] + (left != right)))
        previous = current
    return previous[-1]


def evaluate(model, root):
    report = json.loads((root / 'report.json').read_text(encoding='utf-8'))
    checks = []
    for voice in report['voices']:
        folder = root / voice['folder']
        for output in voice['outputs']:
            cache = folder / (output['file'] + '.asr.json')
            if cache.exists():
                result = json.loads(cache.read_text(encoding='utf-8'))
            else:
                expected = report['texts'][output['text_index'] - 1]
                actual = model._asr_pipe(str(folder / output['file']),
                    generate_kwargs={'language': 'vi', 'task': 'transcribe'})['text']
                words = normalized_words(expected)
                errors = edit_distance(words, normalized_words(actual))
                result = {'voice': voice['name'], 'folder': voice['folder'], 'file': output['file'],
                    'text_index': output['text_index'], 'seed': output['seed'],
                    'expected': expected, 'transcribed': actual, 'word_errors': errors,
                    'word_count': len(words), 'wer': errors / len(words),
                    'review_status': 'needs_listening', 'metric': 'whitespace-token WER, punctuation removed, NFC/casefold'}
                cache.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
            checks.append(result)
            print(f'ASR {len(checks)}: WER={result["wer"]:.3f} — {voice["folder"]}', flush=True)
    metrics = {'asr_model': getattr(model, '_asr_model_name', None),
               'metric_limitations': 'Whisper errors and Vietnamese word spacing affect scores. This is not speaker similarity or a human quality score.',
               'checks': checks}
    (root / 'asr-evaluation.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
    return metrics


if __name__ == '__main__':
    root = Path('/content/clone-pilot')
    evaluate(model, root)
    archive = Path('/content/clone-pilot-reviewed.zip')
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
        for path in root.rglob('*'):
            if path.is_file() and path.suffix != '.pt':
                bundle.write(path, str(path.relative_to(root)))
    import gradio as gr
    if 'pilot_results' in globals():
        pilot_results.close()
    with gr.Blocks(title='Kết quả kiểm tra clone') as reviewed_results:
        gr.Markdown('# Kết quả kiểm tra clone\nBáo cáo ASR là kiểm tra tự động, vẫn cần nghe chất lượng và độ giống giọng.')
        gr.File(value=[str(archive), str(root / 'asr-evaluation.json')], interactive=False)
    reviewed_results.launch(share=True, debug=False)
    print('ASR_COMPLETE', flush=True)
